In [1]:
# Swoop predictive model: match = habit x location x points
# Runs in a DataCamp DataLab notebook (pandas, numpy, scikit-learn are preinstalled).
# Tip: paste each "STEP" block into its own notebook cell.

# =========================================================
# STEP 1: Setup
# =========================================================
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)

CATEGORIES = ["coffee", "food", "grocery", "online"]

# The rewards Maya has in her wallet
REWARDS = pd.DataFrame([
    {"reward": "Free drink",    "merchant": "Campus Brew",  "category": "coffee",  "points": 750,  "dollars": 5.25,  "store_x": 0.30, "store_y": 0.30},
    {"reward": "15% off lunch", "merchant": "Campus Grill", "category": "food",    "points": 0,    "dollars": 1.80,  "store_x": 0.50, "store_y": 0.40},
    {"reward": "$3 off",        "merchant": "Fresh Basket", "category": "grocery", "points": 300,  "dollars": 3.00,  "store_x": 2.00, "store_y": 0.50},
    {"reward": "$10 voucher",   "merchant": "Online store", "category": "online",  "points": 1000, "dollars": 10.00, "store_x": None, "store_y": None},
])

# Places Maya goes (x, y in miles on a simple map)
PLACES = {
    "library":      (0.30, 0.32),
    "union":        (0.50, 0.42),
    "dorms":        (0.45, 0.75),
    "town_center":  (2.00, 0.50),
}

# =========================================================
# STEP 2: Generate 10 weeks of fake student history
# One row per hour (7 AM to 11 PM) per category: did she buy it that hour?
# =========================================================
def where_is_maya(day_of_week, hour):
    weekday = day_of_week < 5
    if weekday:
        if hour < 8:   return "dorms"
        if hour < 11:  return "library"
        if hour < 14:  return "union"
        if hour < 17:  return "library"
        return "dorms"
    if day_of_week == 6 and 14 <= hour < 17:  # Sunday errands
        return "town_center"
    return "dorms"

def true_buy_chance(category, weekday, hour, hours_since):
    """The 'real' behavior the model has to discover. The model never sees this function."""
    if category == "coffee":
        p = 0.45 if (weekday and 8 <= hour <= 10) else 0.04
        if hours_since < 6: p *= 0.2
    elif category == "food":
        p = 0.40 if 11 <= hour <= 13 else (0.18 if 18 <= hour <= 20 else 0.02)
        if hours_since < 3: p *= 0.1
    elif category == "grocery":
        p = 0.08 if (not weekday and 13 <= hour <= 17) else 0.005
        if hours_since > 24 * 5: p *= 4
    else:  # online shopping, mostly late evenings
        p = 0.05 if hour >= 20 else 0.01
    return min(p, 0.95)

rows = []
last_bought = {c: -48 for c in CATEGORIES}  # hours since start
for day in range(70):
    dow = day % 7
    weekday = dow < 5
    for hour in range(7, 23):
        now = day * 24 + hour
        place = where_is_maya(dow, hour)
        for cat in CATEGORIES:
            hours_since = now - last_bought[cat]
            bought = rng.random() < true_buy_chance(cat, weekday, hour, hours_since)
            rows.append({
                "day": day, "day_of_week": dow, "weekday": int(weekday), "hour": hour,
                "place": place, "category": cat,
                "hours_since_last": min(hours_since, 24 * 7),
                "bought": int(bought),
            })
            if bought:
                last_bought[cat] = now

history = pd.DataFrame(rows)
print("Rows of history:", len(history))
print(history.groupby("category")["bought"].sum().rename("purchases"))

# =========================================================
# STEP 3: Turn the raw history into model inputs (features)
# =========================================================
def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["morning"] = df["hour"].between(7, 10).astype(int)
    f["lunch"] = df["hour"].between(11, 13).astype(int)
    f["afternoon"] = df["hour"].between(14, 17).astype(int)
    f["evening"] = df["hour"].between(18, 20).astype(int)
    f["late_night"] = (df["hour"] >= 21).astype(int)
    f["weekday"] = df["weekday"]
    # Combined signals: a weekday morning is a different moment from a weekend morning
    f["weekday_morning"] = f["morning"] * df["weekday"]
    f["weekend_afternoon"] = f["afternoon"] * (1 - df["weekday"])
    f["days_since_last"] = df["hours_since_last"] / 24
    f["bought_recently"] = (df["hours_since_last"] < 4).astype(int)
    return f

FEATURES = list(make_features(history.head(1)).columns)

# =========================================================
# STEP 4: Train one HABIT model per category
# Train on weeks 1-8, test on weeks 9-10 (days the model never saw)
# =========================================================
train = history[history["day"] < 56]
test = history[history["day"] >= 56]

habit_models = {}
print("\nHow well each habit model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)")
for cat in CATEGORIES:
    tr, te = train[train["category"] == cat], test[test["category"] == cat]
    model = LogisticRegression(max_iter=1000)
    model.fit(make_features(tr), tr["bought"])
    habit_models[cat] = model
    auc = roc_auc_score(te["bought"], model.predict_proba(make_features(te))[:, 1])
    print(f"  {cat:8s} AUC = {auc:.2f}")

# =========================================================
# STEP 5: The match score = habit x location x points
# =========================================================
def habit_score(category, day_of_week, hour, hours_since_last):
    row = pd.DataFrame([{"hour": hour, "weekday": int(day_of_week < 5), "hours_since_last": hours_since_last}])
    return float(habit_models[category].predict_proba(make_features(row))[0, 1])

def location_score(place, reward):
    if pd.isna(reward["store_x"]):          # online rewards don't depend on location
        return 1.0
    px, py = PLACES[place]
    miles = float(np.hypot(px - reward["store_x"], py - reward["store_y"]))
    if miles < 0.1: return 1.0               # at the store
    if miles < 0.5: return 0.6               # short walk
    if miles < 1.0: return 0.3
    return 0.1                               # across town

def points_score(reward):
    # Bigger rewards matter more, but never enough to override habit and location.
    # $0 -> 0.5, $5 -> 0.75, $10 or more -> 1.0
    return 0.5 + 0.5 * min(reward["dollars"] / 10, 1.0)

def rank_rewards(day_of_week, hour, place, hours_since):
    out = []
    for _, r in REWARDS.iterrows():
        h = habit_score(r["category"], day_of_week, hour, hours_since[r["category"]])
        l = location_score(place, r)
        p = points_score(r)
        out.append({"merchant": r["merchant"], "reward": r["reward"],
                    "habit": round(h, 2), "location": l, "points": round(p, 2),
                    "match": round(h * l * p, 3)})
    return pd.DataFrame(out).sort_values("match", ascending=False).reset_index(drop=True)

# =========================================================
# STEP 6: Try it on real moments
# =========================================================
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
scenarios = [
    ("Tuesday 8 AM at the library", 1, 8,  "library",     {"coffee": 20, "food": 10, "grocery": 72,  "online": 100}),
    ("Tuesday noon at the union",   1, 12, "union",       {"coffee": 4,  "food": 5,  "grocery": 72,  "online": 100}),
    ("Sunday 3 PM at Town Center",  6, 15, "town_center", {"coffee": 30, "food": 3,  "grocery": 150, "online": 100}),
    ("Wednesday 9 PM in the dorms", 2, 21, "dorms",       {"coffee": 12, "food": 2,  "grocery": 48,  "online": 120}),
]
NOTIFY_AT = 0.10  # only send a notification above this match score

for name, dow, hour, place, since in scenarios:
    ranking = rank_rewards(dow, hour, place, since)
    top = ranking.iloc[0]
    decision = f"NOTIFY: {top['reward']} at {top['merchant']}" if top["match"] >= NOTIFY_AT else "Stay quiet"
    print(f"\n{name} -> {decision}")
    print(ranking.to_string(index=False))

# =========================================================
# STEP 7: Check the whole system on the 2 unseen weeks
# When the model would have notified, did she actually buy that category?
# =========================================================
hits = fires = 0
for (day, hour), slot in test.groupby(["day", "hour"]):
    dow, place = int(slot["day_of_week"].iloc[0]), slot["place"].iloc[0]
    since = dict(zip(slot["category"], slot["hours_since_last"]))
    top = rank_rewards(dow, hour, place, since).iloc[0]
    if top["match"] >= NOTIFY_AT:
        fires += 1
        cat = REWARDS.loc[REWARDS["merchant"] == top["merchant"], "category"].iloc[0]
        hits += int(slot.loc[slot["category"] == cat, "bought"].iloc[0])

base_rate = test["bought"].mean()
print(f"\nNotifications sent in 2 test weeks: {fires}")
print(f"Followed by that purchase: {hits / max(fires, 1):.0%}")
print(f"Chance of a purchase in a random hour and category: {base_rate:.0%}")
print(f"Lift over random: {(hits / max(fires, 1)) / base_rate:.1f}x")

# =========================================================
# STEP 8: Export the trained weights for Lovable
# Copy the printed JSON into your Lovable prompt.
# =========================================================
export = {cat: {"intercept": round(float(m.intercept_[0]), 4),
                "weights": {f: round(float(w), 4) for f, w in zip(FEATURES, m.coef_[0])}}
          for cat, m in habit_models.items()}
print("\nWEIGHTS FOR LOVABLE:")
print(json.dumps(export, indent=2))

Rows of history: 4480
category
coffee     67
food       90
grocery    24
online     13
Name: purchases, dtype: int64

How well each habit model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)
  coffee   AUC = 0.75
  food     AUC = 0.86
  grocery  AUC = 0.72
  online   AUC = 0.92

Tuesday 8 AM at the library -> NOTIFY: Free drink at Campus Brew
    merchant        reward  habit  location  points  match
 Campus Brew    Free drink   0.22       1.0    0.76  0.167
Online store   $10 voucher   0.01       1.0    1.00  0.007
Campus Grill 15% off lunch   0.01       0.6    0.59  0.005
Fresh Basket        $3 off   0.02       0.1    0.65  0.001

Tuesday noon at the union -> NOTIFY: 15% off lunch at Campus Grill
    merchant        reward  habit  location  points  match
Campus Grill 15% off lunch   0.36       1.0    0.59  0.211
 Campus Brew    Free drink   0.03       0.6    0.76  0.013
Online store   $10 voucher   0.01       1.0    1.00  0.006
Fresh Basket        $3 off   0.01       0.1

In [2]:
# Swoop predictive model: match = visit rate x location x points
# Runs in a DataCamp DataLab notebook (pandas, numpy, scikit-learn are preinstalled).
# Tip: paste each "STEP" block into its own notebook cell.

# =========================================================
# STEP 1: Setup
# =========================================================
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)

CATEGORIES = ["coffee", "food", "grocery", "online"]

# The rewards Maya has in her wallet
REWARDS = pd.DataFrame([
    {"reward": "Free drink",    "merchant": "Campus Brew",  "category": "coffee",  "points": 750,  "dollars": 5.25,  "store_x": 0.30, "store_y": 0.30},
    {"reward": "15% off lunch", "merchant": "Campus Grill", "category": "food",    "points": 0,    "dollars": 1.80,  "store_x": 0.50, "store_y": 0.40},
    {"reward": "$3 off",        "merchant": "Fresh Basket", "category": "grocery", "points": 300,  "dollars": 3.00,  "store_x": 2.00, "store_y": 0.50},
    {"reward": "$10 voucher",   "merchant": "Online store", "category": "online",  "points": 1000, "dollars": 10.00, "store_x": None, "store_y": None},
])

# Places Maya goes (x, y in miles on a simple map)
PLACES = {
    "library":      (0.30, 0.32),
    "union":        (0.50, 0.42),
    "dorms":        (0.45, 0.75),
    "town_center":  (2.00, 0.50),
}

# =========================================================
# STEP 2: Generate 10 weeks of fake student history
# One row per hour (7 AM to 11 PM) per category: did she buy it that hour?
# =========================================================
def where_is_maya(day_of_week, hour):
    weekday = day_of_week < 5
    if weekday:
        if hour < 8:   return "dorms"
        if hour < 11:  return "library"
        if hour < 14:  return "union"
        if hour < 17:  return "library"
        return "dorms"
    if day_of_week == 6 and 14 <= hour < 17:  # Sunday errands
        return "town_center"
    return "dorms"

def true_buy_chance(category, weekday, hour, hours_since):
    """The 'real' behavior the model has to discover. The model never sees this function."""
    if category == "coffee":
        p = 0.45 if (weekday and 8 <= hour <= 10) else 0.04
        if hours_since < 6: p *= 0.2
    elif category == "food":
        p = 0.40 if 11 <= hour <= 13 else (0.18 if 18 <= hour <= 20 else 0.02)
        if hours_since < 3: p *= 0.1
    elif category == "grocery":
        p = 0.08 if (not weekday and 13 <= hour <= 17) else 0.005
        if hours_since > 24 * 5: p *= 4
    else:  # online shopping, mostly late evenings
        p = 0.05 if hour >= 20 else 0.01
    return min(p, 0.95)

rows = []
last_bought = {c: -48 for c in CATEGORIES}  # hours since start
for day in range(70):
    dow = day % 7
    weekday = dow < 5
    for hour in range(7, 23):
        now = day * 24 + hour
        place = where_is_maya(dow, hour)
        for cat in CATEGORIES:
            hours_since = now - last_bought[cat]
            bought = rng.random() < true_buy_chance(cat, weekday, hour, hours_since)
            rows.append({
                "day": day, "day_of_week": dow, "weekday": int(weekday), "hour": hour,
                "place": place, "category": cat,
                "hours_since_last": min(hours_since, 24 * 7),
                "bought": int(bought),
            })
            if bought:
                last_bought[cat] = now

history = pd.DataFrame(rows)
print("Rows of history:", len(history))
print(history.groupby("category")["bought"].sum().rename("purchases"))

# =========================================================
# STEP 3: Turn the raw history into model inputs (features)
# =========================================================
# Her USUAL VISIT TIME: the typical hour she buys each category,
# learned separately for weekdays and weekends (training weeks only).
train_purchases = history[(history["day"] < 56) & (history["bought"] == 1)]
USUAL_HOUR = train_purchases.groupby(["category", "weekday"])["hour"].median().to_dict()
print("\nMaya's usual visit times (hour of day):")
for (cat, wd), hr in sorted(USUAL_HOUR.items()):
    print(f"  {cat:8s} {'weekday' if wd else 'weekend'}: {hr:.1f}")

def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["morning"] = df["hour"].between(7, 10).astype(int)
    f["lunch"] = df["hour"].between(11, 13).astype(int)
    f["afternoon"] = df["hour"].between(14, 17).astype(int)
    f["evening"] = df["hour"].between(18, 20).astype(int)
    f["late_night"] = (df["hour"] >= 21).astype(int)
    f["weekday"] = df["weekday"]
    # Combined signals: a weekday morning is a different moment from a weekend morning
    f["weekday_morning"] = f["morning"] * df["weekday"]
    f["weekend_afternoon"] = f["afternoon"] * (1 - df["weekday"])
    # Visit time: how close this hour is to HER usual time for this category
    usual = [USUAL_HOUR.get((c, w), 12) for c, w in zip(df["category"], df["weekday"])]
    gap = (df["hour"] - pd.Series(usual, index=df.index)).abs()
    f["at_usual_time"] = (gap <= 1).astype(int)
    f["hours_from_usual"] = gap / 12
    f["days_since_last"] = df["hours_since_last"] / 24
    f["bought_recently"] = (df["hours_since_last"] < 4).astype(int)
    return f

FEATURES = list(make_features(history.head(1)).columns)

# =========================================================
# STEP 4: Train one VISIT RATE model per category
# Train on weeks 1-8, test on weeks 9-10 (days the model never saw)
# =========================================================
train = history[history["day"] < 56]
test = history[history["day"] >= 56]

habit_models = {}
print("\nHow well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)")
for cat in CATEGORIES:
    tr, te = train[train["category"] == cat], test[test["category"] == cat]
    model = LogisticRegression(max_iter=1000)
    model.fit(make_features(tr), tr["bought"])
    habit_models[cat] = model
    auc = roc_auc_score(te["bought"], model.predict_proba(make_features(te))[:, 1])
    print(f"  {cat:8s} AUC = {auc:.2f}")

# =========================================================
# STEP 5: The match score = visit rate x location x points
# =========================================================
def habit_score(category, day_of_week, hour, hours_since_last):
    row = pd.DataFrame([{"category": category, "hour": hour, "weekday": int(day_of_week < 5), "hours_since_last": hours_since_last}])
    return float(habit_models[category].predict_proba(make_features(row))[0, 1])

def miles_away(place, reward):
    if pd.isna(reward["store_x"]):
        return None                          # online
    px, py = PLACES[place]
    return float(np.hypot(px - reward["store_x"], py - reward["store_y"]))

def location_score(place, reward):
    miles = miles_away(place, reward)
    if miles is None: return 1.0             # online rewards don't depend on location
    if miles < 0.1: return 1.0               # at the store
    if miles < 0.5: return 0.6               # short walk
    if miles < 1.0: return 0.3
    return 0.1                               # across town

def points_score(reward):
    # Bigger rewards matter more, but never enough to override habit and location.
    # $0 -> 0.5, $5 -> 0.75, $10 or more -> 1.0
    return 0.5 + 0.5 * min(reward["dollars"] / 10, 1.0)

def rank_rewards(day_of_week, hour, place, hours_since):
    out = []
    for _, r in REWARDS.iterrows():
        h = habit_score(r["category"], day_of_week, hour, hours_since[r["category"]])
        l = location_score(place, r)
        p = points_score(r)
        m = miles_away(place, r)
        out.append({"merchant": r["merchant"], "reward": r["reward"],
                    "visit_rate": round(h, 2), "miles": "online" if m is None else round(m, 1),
                    "location": l, "points": round(p, 2), "match": round(h * l * p, 3)})
    return pd.DataFrame(out).sort_values("match", ascending=False).reset_index(drop=True)

# =========================================================
# STEP 6: Try it on real moments
# =========================================================
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
scenarios = [
    ("Tuesday 8 AM at the library", 1, 8,  "library",     {"coffee": 20, "food": 10, "grocery": 72,  "online": 100}),
    ("Tuesday noon at the union",   1, 12, "union",       {"coffee": 4,  "food": 5,  "grocery": 72,  "online": 100}),
    ("Sunday 3 PM at Town Center",  6, 15, "town_center", {"coffee": 30, "food": 3,  "grocery": 150, "online": 100}),
    ("Wednesday 9 PM in the dorms", 2, 21, "dorms",       {"coffee": 12, "food": 2,  "grocery": 48,  "online": 120}),
]
NOTIFY_AT = 0.10  # only send a notification above this match score

for name, dow, hour, place, since in scenarios:
    ranking = rank_rewards(dow, hour, place, since)
    top = ranking.iloc[0]
    decision = f"NOTIFY: {top['reward']} at {top['merchant']}" if top["match"] >= NOTIFY_AT else "Stay quiet"
    print(f"\n{name} -> {decision}")
    print(ranking.to_string(index=False))

# =========================================================
# STEP 7: Check the whole system on the 2 unseen weeks
# When the model would have notified, did she actually buy that category?
# =========================================================
hits = fires = 0
for (day, hour), slot in test.groupby(["day", "hour"]):
    dow, place = int(slot["day_of_week"].iloc[0]), slot["place"].iloc[0]
    since = dict(zip(slot["category"], slot["hours_since_last"]))
    top = rank_rewards(dow, hour, place, since).iloc[0]
    if top["match"] >= NOTIFY_AT:
        fires += 1
        cat = REWARDS.loc[REWARDS["merchant"] == top["merchant"], "category"].iloc[0]
        hits += int(slot.loc[slot["category"] == cat, "bought"].iloc[0])

base_rate = test["bought"].mean()
print(f"\nNotifications sent in 2 test weeks: {fires}")
print(f"Followed by that purchase: {hits / max(fires, 1):.0%}")
print(f"Chance of a purchase in a random hour and category: {base_rate:.0%}")
print(f"Lift over random: {(hits / max(fires, 1)) / base_rate:.1f}x")

# =========================================================
# STEP 8: Export the trained weights for Lovable
# Copy the printed JSON into your Lovable prompt.
# =========================================================
export = {cat: {"intercept": round(float(m.intercept_[0]), 4),
                "weights": {f: round(float(w), 4) for f, w in zip(FEATURES, m.coef_[0])}}
          for cat, m in habit_models.items()}
export["usual_hour"] = {f"{c}_{'weekday' if w else 'weekend'}": float(h) for (c, w), h in USUAL_HOUR.items()}
print("\nWEIGHTS FOR LOVABLE:")
print(json.dumps(export, indent=2))

Rows of history: 4480
category
coffee     67
food       90
grocery    24
online     13
Name: purchases, dtype: int64

Maya's usual visit times (hour of day):
  coffee   weekend: 12.5
  coffee   weekday: 9.0
  food     weekend: 13.0
  food     weekday: 12.5
  grocery  weekend: 15.0
  grocery  weekday: 17.0
  online   weekend: 15.0
  online   weekday: 21.0

How well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)
  coffee   AUC = 0.78
  food     AUC = 0.85
  grocery  AUC = 0.73
  online   AUC = 0.93

Tuesday 8 AM at the library -> NOTIFY: Free drink at Campus Brew
    merchant        reward  visit_rate  miles  location  points  match
 Campus Brew    Free drink        0.31    0.0       1.0    0.76  0.233
Campus Grill 15% off lunch        0.01    0.2       0.6    0.59  0.005
Online store   $10 voucher        0.01 online       1.0    1.00  0.005
Fresh Basket        $3 off        0.02    1.7       0.1    0.65  0.001

Tuesday noon at the union -> NOTIFY: 15% 

In [1]:
# Swoop predictive model: match = visit rate x location x points
# Runs in a DataCamp DataLab notebook (pandas, numpy, scikit-learn are preinstalled).
# Tip: paste each "STEP" block into its own notebook cell.

# =========================================================
# STEP 1: Setup
# =========================================================
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)

CATEGORIES = ["coffee", "food", "grocery", "online"]

# The rewards Maya has in her wallet
REWARDS = pd.DataFrame([
    {"reward": "Free drink",    "merchant": "Campus Brew",  "category": "coffee",  "points": 750,  "dollars": 5.25,  "store_x": 0.30, "store_y": 0.30},
    {"reward": "15% off lunch", "merchant": "Campus Grill", "category": "food",    "points": 0,    "dollars": 1.80,  "store_x": 0.50, "store_y": 0.40},
    {"reward": "$8 off order",  "merchant": "Taco Bell",    "category": "food",    "points": 0,    "dollars": 8.00,  "store_x": 0.48, "store_y": 0.70},
    {"reward": "$3 off",        "merchant": "Fresh Basket", "category": "grocery", "points": 300,  "dollars": 3.00,  "store_x": 2.00, "store_y": 0.50},
    {"reward": "$10 voucher",   "merchant": "Online store", "category": "online",  "points": 1000, "dollars": 10.00, "store_x": None, "store_y": None},
])

# Places Maya goes (x, y in miles on a simple map)
PLACES = {
    "library":      (0.30, 0.32),
    "union":        (0.50, 0.42),
    "dorms":        (0.45, 0.75),
    "town_center":  (2.00, 0.50),
}

# =========================================================
# STEP 2: Generate 10 weeks of fake student history
# One row per hour (7 AM to 11 PM) per category: did she buy it that hour?
# =========================================================
def where_is_maya(day_of_week, hour):
    weekday = day_of_week < 5
    if weekday:
        if hour < 8:   return "dorms"
        if hour < 11:  return "library"
        if hour < 14:  return "union"
        if hour < 17:  return "library"
        return "dorms"
    if day_of_week == 6 and 14 <= hour < 17:  # Sunday errands
        return "town_center"
    return "dorms"

def weekday_or_weekend_night(hour, weekday):
    """Late-night food is more common on weekends."""
    return not weekday

def true_buy_chance(category, weekday, hour, hours_since):
    """The 'real' behavior the model has to discover. The model never sees this function."""
    if category == "coffee":
        p = 0.45 if (weekday and 8 <= hour <= 10) else 0.04
        if hours_since < 6: p *= 0.2
    elif category == "food":
        if 11 <= hour <= 13:   p = 0.40          # lunch
        elif 18 <= hour <= 20: p = 0.18          # dinner
        elif hour >= 21:                         # late-night food run (Taco Bell, Burger King)
            p = 0.22 if weekday_or_weekend_night(hour, weekday) else 0.12
        else:                  p = 0.02
        if hours_since < 3: p *= 0.1
    elif category == "grocery":
        p = 0.08 if (not weekday and 13 <= hour <= 17) else 0.005
        if hours_since > 24 * 5: p *= 4
    else:  # online shopping, mostly late evenings
        p = 0.05 if hour >= 20 else 0.01
    return min(p, 0.95)

rows = []
last_bought = {c: -48 for c in CATEGORIES}  # hours since start
for day in range(70):
    dow = day % 7
    weekday = dow < 5
    for hour in range(7, 23):
        now = day * 24 + hour
        place = where_is_maya(dow, hour)
        for cat in CATEGORIES:
            hours_since = now - last_bought[cat]
            bought = rng.random() < true_buy_chance(cat, weekday, hour, hours_since)
            rows.append({
                "day": day, "day_of_week": dow, "weekday": int(weekday), "hour": hour,
                "place": place, "category": cat,
                "hours_since_last": min(hours_since, 24 * 7),
                "bought": int(bought),
            })
            if bought:
                last_bought[cat] = now

history = pd.DataFrame(rows)
print("Rows of history:", len(history))
print(history.groupby("category")["bought"].sum().rename("purchases"))

# =========================================================
# STEP 3: Turn the raw history into model inputs (features)
# =========================================================
# Her USUAL VISIT TIME: the typical hour she buys each category,
# learned separately for weekdays and weekends (training weeks only).
train_purchases = history[(history["day"] < 56) & (history["bought"] == 1)]
USUAL_HOUR = train_purchases.groupby(["category", "weekday"])["hour"].median().to_dict()
print("\nMaya's usual visit times (hour of day):")
for (cat, wd), hr in sorted(USUAL_HOUR.items()):
    print(f"  {cat:8s} {'weekday' if wd else 'weekend'}: {hr:.1f}")

def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["morning"] = df["hour"].between(7, 10).astype(int)
    f["lunch"] = df["hour"].between(11, 13).astype(int)
    f["afternoon"] = df["hour"].between(14, 17).astype(int)
    f["evening"] = df["hour"].between(18, 20).astype(int)
    f["late_night"] = (df["hour"] >= 21).astype(int)
    f["weekday"] = df["weekday"]
    # Combined signals: a weekday morning is a different moment from a weekend morning
    f["weekday_morning"] = f["morning"] * df["weekday"]
    f["weekend_afternoon"] = f["afternoon"] * (1 - df["weekday"])
    # Visit time: how close this hour is to HER usual time for this category
    usual = [USUAL_HOUR.get((c, w), 12) for c, w in zip(df["category"], df["weekday"])]
    gap = (df["hour"] - pd.Series(usual, index=df.index)).abs()
    f["at_usual_time"] = (gap <= 1).astype(int)
    f["hours_from_usual"] = gap / 12
    f["days_since_last"] = df["hours_since_last"] / 24
    f["bought_recently"] = (df["hours_since_last"] < 4).astype(int)
    return f

FEATURES = list(make_features(history.head(1)).columns)

# =========================================================
# STEP 4: Train one VISIT RATE model per category
# Train on weeks 1-8, test on weeks 9-10 (days the model never saw)
# =========================================================
train = history[history["day"] < 56]
test = history[history["day"] >= 56]

habit_models = {}
print("\nHow well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)")
for cat in CATEGORIES:
    tr, te = train[train["category"] == cat], test[test["category"] == cat]
    model = LogisticRegression(max_iter=1000)
    model.fit(make_features(tr), tr["bought"])
    habit_models[cat] = model
    auc = roc_auc_score(te["bought"], model.predict_proba(make_features(te))[:, 1])
    print(f"  {cat:8s} AUC = {auc:.2f}")

# =========================================================
# STEP 5: The match score = visit rate x location x points
# =========================================================
def habit_score(category, day_of_week, hour, hours_since_last):
    row = pd.DataFrame([{"category": category, "hour": hour, "weekday": int(day_of_week < 5), "hours_since_last": hours_since_last}])
    return float(habit_models[category].predict_proba(make_features(row))[0, 1])

def miles_away(place, reward):
    if pd.isna(reward["store_x"]):
        return None                          # online
    px, py = PLACES[place]
    return float(np.hypot(px - reward["store_x"], py - reward["store_y"]))

def location_score(place, reward):
    miles = miles_away(place, reward)
    if miles is None: return 1.0             # online rewards don't depend on location
    if miles < 0.1: return 1.0               # at the store
    if miles < 0.5: return 0.6               # short walk
    if miles < 1.0: return 0.3
    return 0.1                               # across town

def points_score(reward):
    # Bigger rewards matter more, but never enough to override habit and location.
    # $0 -> 0.5, $5 -> 0.75, $10 or more -> 1.0
    return 0.5 + 0.5 * min(reward["dollars"] / 10, 1.0)

def rank_rewards(day_of_week, hour, place, hours_since):
    out = []
    for _, r in REWARDS.iterrows():
        h = habit_score(r["category"], day_of_week, hour, hours_since[r["category"]])
        l = location_score(place, r)
        p = points_score(r)
        m = miles_away(place, r)
        out.append({"merchant": r["merchant"], "reward": r["reward"],
                    "visit_rate": round(h, 2), "miles": "online" if m is None else round(m, 1),
                    "location": l, "points": round(p, 2), "match": round(h * l * p, 3)})
    return pd.DataFrame(out).sort_values("match", ascending=False).reset_index(drop=True)

# =========================================================
# STEP 6: Try it on real moments
# =========================================================
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
scenarios = [
    ("Tuesday 8 AM at the library", 1, 8,  "library",     {"coffee": 20, "food": 10, "grocery": 72,  "online": 100}),
    ("Tuesday noon at the union",   1, 12, "union",       {"coffee": 4,  "food": 5,  "grocery": 72,  "online": 100}),
    ("Sunday 3 PM at Town Center",  6, 15, "town_center", {"coffee": 30, "food": 3,  "grocery": 150, "online": 100}),
    ("Wednesday 9 PM in the dorms", 2, 21, "dorms",       {"coffee": 12, "food": 2,  "grocery": 48,  "online": 120}),
    ("Wednesday 10 PM in the dorms", 2, 22, "dorms",      {"coffee": 13, "food": 4,  "grocery": 48,  "online": 120}),
    ("Saturday 10 PM in the dorms", 5, 22, "dorms",       {"coffee": 14, "food": 4,  "grocery": 48,  "online": 120}),
    ("Wednesday 3 PM in the dorms", 2, 15, "dorms",       {"coffee": 6,  "food": 2,  "grocery": 24,  "online": 30}),
]
NOTIFY_AT = 0.115  # only send a notification above this match score

for name, dow, hour, place, since in scenarios:
    ranking = rank_rewards(dow, hour, place, since)
    top = ranking.iloc[0]
    decision = f"NOTIFY: {top['reward']} at {top['merchant']}" if top["match"] >= NOTIFY_AT else "Stay quiet"
    print(f"\n{name} -> {decision}")
    print(ranking.to_string(index=False))

# =========================================================
# STEP 7: Check the whole system on the 2 unseen weeks
# When the model would have notified, did she actually buy that category?
# =========================================================
hits = fires = 0
for (day, hour), slot in test.groupby(["day", "hour"]):
    dow, place = int(slot["day_of_week"].iloc[0]), slot["place"].iloc[0]
    since = dict(zip(slot["category"], slot["hours_since_last"]))
    top = rank_rewards(dow, hour, place, since).iloc[0]
    if top["match"] >= NOTIFY_AT:
        fires += 1
        cat = REWARDS.loc[REWARDS["merchant"] == top["merchant"], "category"].iloc[0]
        hits += int(slot.loc[slot["category"] == cat, "bought"].iloc[0])

base_rate = test["bought"].mean()
print(f"\nNotifications sent in 2 test weeks: {fires}")
print(f"Followed by that purchase: {hits / max(fires, 1):.0%}")
print(f"Chance of a purchase in a random hour and category: {base_rate:.0%}")
print(f"Lift over random: {(hits / max(fires, 1)) / base_rate:.1f}x")

# =========================================================
# STEP 8: Export the trained weights for Lovable
# Copy the printed JSON into your Lovable prompt.
# =========================================================
export = {cat: {"intercept": round(float(m.intercept_[0]), 4),
                "weights": {f: round(float(w), 4) for f, w in zip(FEATURES, m.coef_[0])}}
          for cat, m in habit_models.items()}
export["usual_hour"] = {f"{c}_{'weekday' if w else 'weekend'}": float(h) for (c, w), h in USUAL_HOUR.items()}
print("\nWEIGHTS FOR LOVABLE:")
print(json.dumps(export, indent=2))

# =========================================================
# STEP 9: Export Maya's fake purchase history for the app
# The app uses this for weekly counts, "last bought" times, and history.
# Each purchase: [days_ago, hour, category, merchant]
# =========================================================
def app_merchant(category, hour, day):
    if category == "coffee":  return "Starbucks"
    if category == "online":  return "Amazon"
    if category == "grocery": return "Grocery store"
    if hour >= 21:            return "Taco Bell" if day % 2 else "Burger King"
    if 11 <= hour <= 14:      return "Chick-fil-A" if day % 2 else "Chipotle"
    return "Chick-fil-A" if day % 3 else "McDonald's"

last_day = history["day"].max()
purchases = history[history["bought"] == 1].sort_values(["day", "hour"])
maya_history = [[int(last_day - r.day), int(r.hour), r.category, app_merchant(r.category, r.hour, r.day)]
                for r in purchases.itertuples()]
print(f"\nMAYA'S HISTORY FOR LOVABLE ({len(maya_history)} purchases over 10 weeks):")
print(json.dumps(maya_history, separators=(",", ":")))

weekly = purchases[purchases["day"] > last_day - 28].groupby("category").size() / 4
print("\nAverage purchases per week (last 4 weeks):")
print(weekly.round(1).to_string())

Rows of history: 4480
category
coffee      67
food       112
grocery     24
online      13
Name: purchases, dtype: int64

Maya's usual visit times (hour of day):
  coffee   weekend: 12.5
  coffee   weekday: 9.0
  food     weekend: 18.5
  food     weekday: 13.0
  grocery  weekend: 15.0
  grocery  weekday: 17.0
  online   weekend: 15.0
  online   weekday: 21.0

How well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)
  coffee   AUC = 0.78
  food     AUC = 0.83
  grocery  AUC = 0.73
  online   AUC = 0.93

Tuesday 8 AM at the library -> NOTIFY: Free drink at Campus Brew
    merchant        reward  visit_rate  miles  location  points  match
 Campus Brew    Free drink        0.31    0.0       1.0    0.76  0.233
   Taco Bell  $8 off order        0.01    0.4       0.6    0.90  0.008
Campus Grill 15% off lunch        0.01    0.2       0.6    0.59  0.005
Online store   $10 voucher        0.01 online       1.0    1.00  0.005
Fresh Basket        $3 off        0.02

In [2]:
# Swoop predictive model: match = visit rate x location x points
# Runs in a DataCamp DataLab notebook (pandas, numpy, scikit-learn are preinstalled).
# Tip: paste each "STEP" block into its own notebook cell.

# =========================================================
# STEP 1: Setup
# =========================================================
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
mood_rng = np.random.default_rng(7)   # separate random numbers for her weekly spending mood

CATEGORIES = ["coffee", "food", "grocery", "online"]

# The rewards Sarah has in her wallet
REWARDS = pd.DataFrame([
    {"reward": "Free drink",    "merchant": "Campus Brew",  "category": "coffee",  "points": 750,  "dollars": 5.25,  "store_x": 0.30, "store_y": 0.30},
    {"reward": "15% off lunch", "merchant": "Campus Grill", "category": "food",    "points": 0,    "dollars": 1.80,  "store_x": 0.50, "store_y": 0.40},
    {"reward": "$8 off order",  "merchant": "Taco Bell",    "category": "food",    "points": 0,    "dollars": 8.00,  "store_x": 0.48, "store_y": 0.70},
    {"reward": "$3 off",        "merchant": "Fresh Basket", "category": "grocery", "points": 300,  "dollars": 3.00,  "store_x": 2.00, "store_y": 0.50},
    {"reward": "$10 voucher",   "merchant": "Online store", "category": "online",  "points": 1000, "dollars": 10.00, "store_x": None, "store_y": None},
])

# Places Sarah goes (x, y in miles on a simple map)
PLACES = {
    "library":      (0.30, 0.32),
    "union":        (0.50, 0.42),
    "dorms":        (0.45, 0.75),
    "town_center":  (2.00, 0.50),
}

# =========================================================
# STEP 2: Generate 10 weeks of fake student history
# One row per hour (7 AM to 11 PM) per category: did she buy it that hour?
# =========================================================
def where_is_sarah(day_of_week, hour):
    weekday = day_of_week < 5
    if weekday:
        if hour < 8:   return "dorms"
        if hour < 11:  return "library"
        if hour < 14:  return "union"
        if hour < 17:  return "library"
        return "dorms"
    if day_of_week == 6 and 14 <= hour < 17:  # Sunday errands
        return "town_center"
    return "dorms"

def weekday_or_weekend_night(hour, weekday):
    """Late-night food is more common on weekends."""
    return not weekday

def true_buy_chance(category, weekday, hour, hours_since, week_mood):
    """The 'real' behavior the model has to discover. The model never sees this function.
    week_mood: some weeks she spends freely (1.4), some weeks money is tight (0.6)."""
    if category == "coffee":
        p = 0.45 if (weekday and 8 <= hour <= 10) else 0.04
        if hours_since < 6: p *= 0.2
    elif category == "food":
        if 11 <= hour <= 13:   p = 0.40          # lunch
        elif 18 <= hour <= 20: p = 0.18          # dinner
        elif hour >= 21:                         # late-night food run (Taco Bell, Burger King)
            p = 0.22 if weekday_or_weekend_night(hour, weekday) else 0.12
        else:                  p = 0.02
        if hours_since < 3: p *= 0.1
    elif category == "grocery":
        p = 0.08 if (not weekday and 13 <= hour <= 17) else 0.005
        if hours_since > 24 * 5: p *= 4
    else:  # online shopping, mostly late evenings
        p = 0.05 if hour >= 20 else 0.01
    if category != "grocery":      # coffee, eating out, and online shopping follow her weekly mood
        p *= week_mood
    return min(p, 0.95)

def spend_velocity(purchase_times, now):
    """Purchases in the last 7 days divided by her average week over the last 28 days.
    1.0 = normal week, above 1 = spending faster than usual, below 1 = tighter week."""
    last_7 = sum(1 for t in purchase_times if now - 168 <= t < now)
    window = min(now, 672)                       # up to 28 days of history
    last_28 = sum(1 for t in purchase_times if now - window <= t < now)
    if window < 168 or last_28 == 0:
        return 1.0
    weekly_avg = last_28 / (window / 168)
    return min(last_7 / weekly_avg, 2.5)

rows = []
last_bought = {c: -48 for c in CATEGORIES}  # hours since start
purchase_times = []                          # every purchase, any category
week_mood = 1.0
for day in range(70):
    dow = day % 7
    weekday = dow < 5
    if day % 14 == 0:                        # every other Monday, a new spending mood (lasts 2 weeks)
        week_mood = float(mood_rng.uniform(0.5, 1.5))
    for hour in range(7, 23):
        now = day * 24 + hour
        place = where_is_sarah(dow, hour)
        velocity = spend_velocity(purchase_times, now)
        for cat in CATEGORIES:
            hours_since = now - last_bought[cat]
            bought = rng.random() < true_buy_chance(cat, weekday, hour, hours_since, week_mood)
            rows.append({
                "day": day, "day_of_week": dow, "weekday": int(weekday), "hour": hour,
                "place": place, "category": cat,
                "hours_since_last": min(hours_since, 24 * 7),
                "spend_velocity": round(velocity, 3),
                "bought": int(bought),
            })
            if bought:
                last_bought[cat] = now
                purchase_times.append(now)

history = pd.DataFrame(rows)
print("Rows of history:", len(history))
print(history.groupby("category")["bought"].sum().rename("purchases"))

# =========================================================
# STEP 3: Turn the raw history into model inputs (features)
# =========================================================
# Her USUAL VISIT TIME: the typical hour she buys each category,
# learned separately for weekdays and weekends (training weeks only).
train_purchases = history[(history["day"] < 56) & (history["bought"] == 1)]
USUAL_HOUR = train_purchases.groupby(["category", "weekday"])["hour"].median().to_dict()
print("\nSarah's usual visit times (hour of day):")
for (cat, wd), hr in sorted(USUAL_HOUR.items()):
    print(f"  {cat:8s} {'weekday' if wd else 'weekend'}: {hr:.1f}")

def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["morning"] = df["hour"].between(7, 10).astype(int)
    f["lunch"] = df["hour"].between(11, 13).astype(int)
    f["afternoon"] = df["hour"].between(14, 17).astype(int)
    f["evening"] = df["hour"].between(18, 20).astype(int)
    f["late_night"] = (df["hour"] >= 21).astype(int)
    f["weekday"] = df["weekday"]
    # Combined signals: a weekday morning is a different moment from a weekend morning
    f["weekday_morning"] = f["morning"] * df["weekday"]
    f["weekend_afternoon"] = f["afternoon"] * (1 - df["weekday"])
    # Visit time: how close this hour is to HER usual time for this category
    usual = [USUAL_HOUR.get((c, w), 12) for c, w in zip(df["category"], df["weekday"])]
    gap = (df["hour"] - pd.Series(usual, index=df.index)).abs()
    f["at_usual_time"] = (gap <= 1).astype(int)
    f["hours_from_usual"] = gap / 12
    # Spend velocity: is she spending faster (+) or slower (-) than her usual week?
    f["spend_velocity"] = df["spend_velocity"] - 1
    f["days_since_last"] = df["hours_since_last"] / 24
    f["bought_recently"] = (df["hours_since_last"] < 4).astype(int)
    return f

FEATURES = list(make_features(history.head(1)).columns)

# =========================================================
# STEP 4: Train one VISIT RATE model per category
# Train on weeks 1-8, test on weeks 9-10 (days the model never saw)
# =========================================================
train = history[history["day"] < 56]
test = history[history["day"] >= 56]

habit_models = {}
print("\nHow well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)")
for cat in CATEGORIES:
    tr, te = train[train["category"] == cat], test[test["category"] == cat]
    model = LogisticRegression(max_iter=1000)
    model.fit(make_features(tr), tr["bought"])
    habit_models[cat] = model
    auc = roc_auc_score(te["bought"], model.predict_proba(make_features(te))[:, 1])
    print(f"  {cat:8s} AUC = {auc:.2f}")

# =========================================================
# STEP 5: The match score = visit rate x location x points
# =========================================================
def habit_score(category, day_of_week, hour, hours_since_last, velocity=1.0):
    row = pd.DataFrame([{"category": category, "hour": hour, "weekday": int(day_of_week < 5),
                         "hours_since_last": hours_since_last, "spend_velocity": velocity}])
    return float(habit_models[category].predict_proba(make_features(row))[0, 1])

def miles_away(place, reward):
    if pd.isna(reward["store_x"]):
        return None                          # online
    px, py = PLACES[place]
    return float(np.hypot(px - reward["store_x"], py - reward["store_y"]))

def location_score(place, reward):
    miles = miles_away(place, reward)
    if miles is None: return 1.0             # online rewards don't depend on location
    if miles < 0.1: return 1.0               # at the store
    if miles < 0.5: return 0.6               # short walk
    if miles < 1.0: return 0.3
    return 0.1                               # across town

def points_score(reward):
    # Bigger rewards matter more, but never enough to override habit and location.
    # $0 -> 0.5, $5 -> 0.75, $10 or more -> 1.0
    return 0.5 + 0.5 * min(reward["dollars"] / 10, 1.0)

def rank_rewards(day_of_week, hour, place, hours_since, velocity=1.0):
    out = []
    for _, r in REWARDS.iterrows():
        h = habit_score(r["category"], day_of_week, hour, hours_since[r["category"]], velocity)
        l = location_score(place, r)
        p = points_score(r)
        m = miles_away(place, r)
        out.append({"merchant": r["merchant"], "reward": r["reward"],
                    "visit_rate": round(h, 2), "miles": "online" if m is None else round(m, 1),
                    "location": l, "points": round(p, 2), "match": round(h * l * p, 3)})
    return pd.DataFrame(out).sort_values("match", ascending=False).reset_index(drop=True)

# =========================================================
# STEP 6: Try it on real moments
# =========================================================
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
scenarios = [
    ("Tuesday 8 AM at the library", 1, 8,  "library",     {"coffee": 20, "food": 10, "grocery": 72,  "online": 100}),
    ("Tuesday noon at the union",   1, 12, "union",       {"coffee": 4,  "food": 5,  "grocery": 72,  "online": 100}),
    ("Sunday 3 PM at Town Center",  6, 15, "town_center", {"coffee": 30, "food": 3,  "grocery": 150, "online": 100}),
    ("Wednesday 9 PM in the dorms", 2, 21, "dorms",       {"coffee": 12, "food": 2,  "grocery": 48,  "online": 120}),
    ("Wednesday 10 PM in the dorms", 2, 22, "dorms",      {"coffee": 13, "food": 4,  "grocery": 48,  "online": 120}),
    ("Saturday 10 PM in the dorms", 5, 22, "dorms",       {"coffee": 14, "food": 4,  "grocery": 48,  "online": 120}),
    ("Wednesday 3 PM in the dorms", 2, 15, "dorms",       {"coffee": 6,  "food": 2,  "grocery": 24,  "online": 30}),
]
NOTIFY_AT = 0.115  # only send a notification above this match score

for name, dow, hour, place, since in scenarios:
    ranking = rank_rewards(dow, hour, place, since)
    top = ranking.iloc[0]
    decision = f"NOTIFY: {top['reward']} at {top['merchant']}" if top["match"] >= NOTIFY_AT else "Stay quiet"
    print(f"\n{name} -> {decision}")
    print(ranking.to_string(index=False))

# Same moment, different spending weeks: shows the model learned spend velocity
print("\nLikely now on Tuesday at noon (lunch), by spending pace this week:")
for v, label in [(0.6, "tight week"), (1.0, "normal week"), (1.4, "spending more")]:
    print(f"  {label:14s} (velocity {v}): food {habit_score('food', 1, 12, 5, v):.0%}, coffee at 8 AM {habit_score('coffee', 1, 8, 20, v):.0%}")

# =========================================================
# STEP 7: Check the whole system on the 2 unseen weeks
# When the model would have notified, did she actually buy that category?
# =========================================================
hits = fires = 0
for (day, hour), slot in test.groupby(["day", "hour"]):
    dow, place = int(slot["day_of_week"].iloc[0]), slot["place"].iloc[0]
    since = dict(zip(slot["category"], slot["hours_since_last"]))
    top = rank_rewards(dow, hour, place, since, float(slot["spend_velocity"].iloc[0])).iloc[0]
    if top["match"] >= NOTIFY_AT:
        fires += 1
        cat = REWARDS.loc[REWARDS["merchant"] == top["merchant"], "category"].iloc[0]
        hits += int(slot.loc[slot["category"] == cat, "bought"].iloc[0])

base_rate = test["bought"].mean()
print(f"\nNotifications sent in 2 test weeks: {fires}")
print(f"Followed by that purchase: {hits / max(fires, 1):.0%}")
print(f"Chance of a purchase in a random hour and category: {base_rate:.0%}")
print(f"Lift over random: {(hits / max(fires, 1)) / base_rate:.1f}x")

# =========================================================
# STEP 8: Export the trained weights for Lovable
# Copy the printed JSON into your Lovable prompt.
# =========================================================
export = {cat: {"intercept": round(float(m.intercept_[0]), 4),
                "weights": {f: round(float(w), 4) for f, w in zip(FEATURES, m.coef_[0])}}
          for cat, m in habit_models.items()}
export["usual_hour"] = {f"{c}_{'weekday' if w else 'weekend'}": float(h) for (c, w), h in USUAL_HOUR.items()}
print("\nWEIGHTS FOR LOVABLE:")
print(json.dumps(export, indent=2))

# =========================================================
# STEP 9: Export Sarah's fake purchase history for the app
# The app uses this for weekly counts, "last bought" times, and history.
# Each purchase: [days_ago, hour, category, merchant]
# =========================================================
def app_merchant(category, hour, day):
    if category == "coffee":  return "Starbucks"
    if category == "online":  return "Amazon"
    if category == "grocery": return "Grocery store"
    if hour >= 21:            return "Taco Bell" if day % 2 else "Burger King"
    if 11 <= hour <= 14:      return "Chick-fil-A" if day % 2 else "Chipotle"
    return "Chick-fil-A" if day % 3 else "McDonald's"

last_day = history["day"].max()
purchases = history[history["bought"] == 1].sort_values(["day", "hour"])
sarah_history = [[int(last_day - r.day), int(r.hour), r.category, app_merchant(r.category, r.hour, r.day)]
                for r in purchases.itertuples()]
print(f"\nMAYA'S HISTORY FOR LOVABLE ({len(sarah_history)} purchases over 10 weeks):")
print(json.dumps(sarah_history, separators=(",", ":")))

weekly = purchases[purchases["day"] > last_day - 28].groupby("category").size() / 4
print("\nAverage purchases per week (last 4 weeks):")
print(weekly.round(1).to_string())

Rows of history: 4480
category
coffee      77
food       119
grocery     24
online      14
Name: purchases, dtype: int64

Sarah's usual visit times (hour of day):
  coffee   weekend: 13.0
  coffee   weekday: 9.0
  food     weekend: 15.0
  food     weekday: 13.0
  grocery  weekend: 15.0
  grocery  weekday: 17.0
  online   weekend: 15.0
  online   weekday: 20.0

How well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)
  coffee   AUC = 0.73
  food     AUC = 0.83
  grocery  AUC = 0.73
  online   AUC = 1.00

Tuesday 8 AM at the library -> NOTIFY: Free drink at Campus Brew
    merchant        reward  visit_rate  miles  location  points  match
 Campus Brew    Free drink        0.45    0.0       1.0    0.76  0.341
   Taco Bell  $8 off order        0.02    0.4       0.6    0.90  0.012
Campus Grill 15% off lunch        0.02    0.2       0.6    0.59  0.008
Online store   $10 voucher        0.01 online       1.0    1.00  0.006
Fresh Basket        $3 off        0.0

In [1]:
# Swoop predictive model: match = visit rate x location x points
# Runs in a DataCamp DataLab notebook (pandas, numpy, scikit-learn are preinstalled).
# Tip: paste each "STEP" block into its own notebook cell.

# =========================================================
# STEP 1: Setup
# =========================================================
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
mood_rng = np.random.default_rng(7)   # separate random numbers for her weekly spending mood
ride_rng = np.random.default_rng(99)  # separate random numbers for rides, so adding them leaves the rest of her history unchanged

CATEGORIES = ["coffee", "food", "grocery", "online", "ride"]

# The rewards Sarah has in her wallet
REWARDS = pd.DataFrame([
    {"reward": "Free drink",    "merchant": "Campus Brew",  "category": "coffee",  "points": 750,  "dollars": 5.25,  "store_x": 0.30, "store_y": 0.30},
    {"reward": "15% off lunch", "merchant": "Campus Grill", "category": "food",    "points": 0,    "dollars": 1.80,  "store_x": 0.50, "store_y": 0.40},
    {"reward": "$8 off order",  "merchant": "Taco Bell",    "category": "food",    "points": 0,    "dollars": 8.00,  "store_x": 0.48, "store_y": 0.70},
    {"reward": "$3 off",        "merchant": "Fresh Basket", "category": "grocery", "points": 300,  "dollars": 3.00,  "store_x": 2.00, "store_y": 0.50},
    {"reward": "$10 voucher",   "merchant": "Online store", "category": "online",  "points": 1000, "dollars": 10.00, "store_x": None, "store_y": None},
    {"reward": "$4 off a ride", "merchant": "Lyft",         "category": "ride",    "points": 400,  "dollars": 4.00,  "store_x": None, "store_y": None},  # rides come to her, so no store distance
])

# Places Sarah goes (x, y in miles on a simple map)
PLACES = {
    "library":      (0.30, 0.32),
    "union":        (0.50, 0.42),
    "dorms":        (0.45, 0.75),
    "town_center":  (2.00, 0.50),
}

# =========================================================
# STEP 2: Generate 10 weeks of fake student history
# One row per hour (7 AM to 11 PM) per category: did she buy it that hour?
# =========================================================
def where_is_sarah(day_of_week, hour):
    weekday = day_of_week < 5
    if weekday:
        if hour < 8:   return "dorms"
        if hour < 11:  return "library"
        if hour < 14:  return "union"
        if hour < 17:  return "library"
        return "dorms"
    if day_of_week == 6 and 14 <= hour < 17:  # Sunday errands
        return "town_center"
    return "dorms"

def weekday_or_weekend_night(hour, weekday):
    """Late-night food is more common on weekends."""
    return not weekday

def true_buy_chance(category, weekday, hour, hours_since, week_mood):
    """The 'real' behavior the model has to discover. The model never sees this function.
    week_mood: some weeks she spends freely (1.4), some weeks money is tight (0.6)."""
    if category == "coffee":
        p = 0.45 if (weekday and 8 <= hour <= 10) else 0.04
        if hours_since < 6: p *= 0.2
    elif category == "food":
        if 11 <= hour <= 13:   p = 0.40          # lunch
        elif 18 <= hour <= 20: p = 0.18          # dinner
        elif hour >= 21:                         # late-night food run (Taco Bell, Burger King)
            p = 0.22 if weekday_or_weekend_night(hour, weekday) else 0.12
        else:                  p = 0.02
        if hours_since < 3: p *= 0.1
    elif category == "grocery":
        p = 0.08 if (not weekday and 13 <= hour <= 17) else 0.005
        if hours_since > 24 * 5: p *= 4
    elif category == "ride":  # a ride home when classes end on weekdays, late nights out on weekends
        if weekday and 13 <= hour <= 17: p = 0.26 if hours_since > 12 else 0.02
        elif not weekday and hour >= 21: p = 0.12
        else:                            p = 0.004
    else:  # online shopping, mostly late evenings
        p = 0.05 if hour >= 20 else 0.01
    if category != "grocery":      # coffee, eating out, and online shopping follow her weekly mood
        p *= week_mood
    return min(p, 0.95)

def spend_velocity(purchase_times, now):
    """Purchases in the last 7 days divided by her average week over the last 28 days.
    1.0 = normal week, above 1 = spending faster than usual, below 1 = tighter week."""
    last_7 = sum(1 for t in purchase_times if now - 168 <= t < now)
    window = min(now, 672)                       # up to 28 days of history
    last_28 = sum(1 for t in purchase_times if now - window <= t < now)
    if window < 168 or last_28 == 0:
        return 1.0
    weekly_avg = last_28 / (window / 168)
    return min(last_7 / weekly_avg, 2.5)

rows = []
last_bought = {c: -48 for c in CATEGORIES}  # hours since start
purchase_times = []                          # every purchase, any category
week_mood = 1.0
for day in range(70):
    dow = day % 7
    weekday = dow < 5
    if day % 14 == 0:                        # every other Monday, a new spending mood (lasts 2 weeks)
        week_mood = float(mood_rng.uniform(0.5, 1.5))
    for hour in range(7, 23):
        now = day * 24 + hour
        place = where_is_sarah(dow, hour)
        velocity = spend_velocity(purchase_times, now)
        for cat in CATEGORIES:
            hours_since = now - last_bought[cat]
            draw = ride_rng.random() if cat == "ride" else rng.random()
            bought = draw < true_buy_chance(cat, weekday, hour, hours_since, week_mood)
            rows.append({
                "day": day, "day_of_week": dow, "weekday": int(weekday), "hour": hour,
                "place": place, "category": cat,
                "hours_since_last": min(hours_since, 24 * 7),
                "spend_velocity": round(velocity, 3),
                "bought": int(bought),
            })
            if bought:
                last_bought[cat] = now
                purchase_times.append(now)

history = pd.DataFrame(rows)
print("Rows of history:", len(history))
print(history.groupby("category")["bought"].sum().rename("purchases"))

# =========================================================
# STEP 3: Turn the raw history into model inputs (features)
# =========================================================
# Her USUAL VISIT TIME: the typical hour she buys each category,
# learned separately for weekdays and weekends (training weeks only).
train_purchases = history[(history["day"] < 56) & (history["bought"] == 1)]
USUAL_HOUR = train_purchases.groupby(["category", "weekday"])["hour"].median().to_dict()
print("\nSarah's usual visit times (hour of day):")
for (cat, wd), hr in sorted(USUAL_HOUR.items()):
    print(f"  {cat:8s} {'weekday' if wd else 'weekend'}: {hr:.1f}")

def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["morning"] = df["hour"].between(7, 10).astype(int)
    f["lunch"] = df["hour"].between(11, 13).astype(int)
    f["afternoon"] = df["hour"].between(14, 17).astype(int)
    f["evening"] = df["hour"].between(18, 20).astype(int)
    f["late_night"] = (df["hour"] >= 21).astype(int)
    f["weekday"] = df["weekday"]
    # Combined signals: a weekday morning is a different moment from a weekend morning
    f["weekday_morning"] = f["morning"] * df["weekday"]
    f["weekend_afternoon"] = f["afternoon"] * (1 - df["weekday"])
    # Visit time: how close this hour is to HER usual time for this category
    usual = [USUAL_HOUR.get((c, w), 12) for c, w in zip(df["category"], df["weekday"])]
    gap = (df["hour"] - pd.Series(usual, index=df.index)).abs()
    f["at_usual_time"] = (gap <= 1).astype(int)
    f["hours_from_usual"] = gap / 12
    # Spend velocity: is she spending faster (+) or slower (-) than her usual week?
    f["spend_velocity"] = df["spend_velocity"] - 1
    f["days_since_last"] = df["hours_since_last"] / 24
    f["bought_recently"] = (df["hours_since_last"] < 4).astype(int)
    return f

FEATURES = list(make_features(history.head(1)).columns)

# =========================================================
# STEP 4: Train one VISIT RATE model per category
# Train on weeks 1-8, test on weeks 9-10 (days the model never saw)
# =========================================================
train = history[history["day"] < 56]
test = history[history["day"] >= 56]

habit_models = {}
print("\nHow well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)")
for cat in CATEGORIES:
    tr, te = train[train["category"] == cat], test[test["category"] == cat]
    model = LogisticRegression(max_iter=1000)
    model.fit(make_features(tr), tr["bought"])
    habit_models[cat] = model
    auc = roc_auc_score(te["bought"], model.predict_proba(make_features(te))[:, 1])
    print(f"  {cat:8s} AUC = {auc:.2f}")

# =========================================================
# STEP 5: The match score = visit rate x location x points
# =========================================================
def habit_score(category, day_of_week, hour, hours_since_last, velocity=1.0):
    row = pd.DataFrame([{"category": category, "hour": hour, "weekday": int(day_of_week < 5),
                         "hours_since_last": hours_since_last, "spend_velocity": velocity}])
    return float(habit_models[category].predict_proba(make_features(row))[0, 1])

def miles_away(place, reward):
    if pd.isna(reward["store_x"]):
        return None                          # online
    px, py = PLACES[place]
    return float(np.hypot(px - reward["store_x"], py - reward["store_y"]))

def location_score(place, reward):
    miles = miles_away(place, reward)
    if miles is None: return 1.0             # online rewards don't depend on location
    if miles < 0.1: return 1.0               # at the store
    if miles < 0.5: return 0.6               # short walk
    if miles < 1.0: return 0.3
    return 0.1                               # across town

def points_score(reward):
    # Bigger rewards matter more, but never enough to override habit and location.
    # $0 -> 0.5, $5 -> 0.75, $10 or more -> 1.0
    return 0.5 + 0.5 * min(reward["dollars"] / 10, 1.0)

def rank_rewards(day_of_week, hour, place, hours_since, velocity=1.0):
    out = []
    for _, r in REWARDS.iterrows():
        h = habit_score(r["category"], day_of_week, hour, hours_since[r["category"]], velocity)
        l = location_score(place, r)
        p = points_score(r)
        m = miles_away(place, r)
        out.append({"merchant": r["merchant"], "reward": r["reward"],
                    "visit_rate": round(h, 2), "miles": "online" if m is None else round(m, 1),
                    "location": l, "points": round(p, 2), "match": round(h * l * p, 3)})
    return pd.DataFrame(out).sort_values("match", ascending=False).reset_index(drop=True)

# =========================================================
# STEP 6: Try it on real moments
# =========================================================
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
scenarios = [
    ("Tuesday 8 AM at the library", 1, 8,  "library",     {"coffee": 20, "food": 10, "grocery": 72,  "online": 100, "ride": 22}),
    ("Tuesday noon at the union",   1, 12, "union",       {"coffee": 4,  "food": 5,  "grocery": 72,  "online": 100, "ride": 22}),
    ("Sunday 3 PM at Town Center",  6, 15, "town_center", {"coffee": 30, "food": 3,  "grocery": 150, "online": 100, "ride": 22}),
    ("Wednesday 9 PM in the dorms", 2, 21, "dorms",       {"coffee": 12, "food": 2,  "grocery": 48,  "online": 120, "ride": 22}),
    ("Wednesday 10 PM in the dorms", 2, 22, "dorms",      {"coffee": 13, "food": 4,  "grocery": 48,  "online": 120, "ride": 22}),
    ("Saturday 10 PM in the dorms", 5, 22, "dorms",       {"coffee": 14, "food": 4,  "grocery": 48,  "online": 120, "ride": 22}),
    ("Tuesday 4 PM at the library",   1, 16, "library",     {"coffee": 8,  "food": 4,  "grocery": 72,  "online": 100, "ride": 23}),
    ("Wednesday 3 PM in the dorms", 2, 15, "dorms",       {"coffee": 6,  "food": 2,  "grocery": 24,  "online": 30, "ride": 22}),
]
NOTIFY_AT = 0.115  # only send a notification above this match score

for name, dow, hour, place, since in scenarios:
    ranking = rank_rewards(dow, hour, place, since)
    top = ranking.iloc[0]
    decision = f"NOTIFY: {top['reward']} at {top['merchant']}" if top["match"] >= NOTIFY_AT else "Stay quiet"
    print(f"\n{name} -> {decision}")
    print(ranking.to_string(index=False))

# Same moment, different spending weeks: shows the model learned spend velocity
print("\nLikely now on Tuesday at noon (lunch), by spending pace this week:")
for v, label in [(0.6, "tight week"), (1.0, "normal week"), (1.4, "spending more")]:
    print(f"  {label:14s} (velocity {v}): food {habit_score('food', 1, 12, 5, v):.0%}, coffee at 8 AM {habit_score('coffee', 1, 8, 20, v):.0%}")

# =========================================================
# STEP 7: Check the whole system on the 2 unseen weeks
# When the model would have notified, did she actually buy that category?
# =========================================================
hits = fires = 0
for (day, hour), slot in test.groupby(["day", "hour"]):
    dow, place = int(slot["day_of_week"].iloc[0]), slot["place"].iloc[0]
    since = dict(zip(slot["category"], slot["hours_since_last"]))
    top = rank_rewards(dow, hour, place, since, float(slot["spend_velocity"].iloc[0])).iloc[0]
    if top["match"] >= NOTIFY_AT:
        fires += 1
        cat = REWARDS.loc[REWARDS["merchant"] == top["merchant"], "category"].iloc[0]
        hits += int(slot.loc[slot["category"] == cat, "bought"].iloc[0])

base_rate = test["bought"].mean()
print(f"\nNotifications sent in 2 test weeks: {fires}")
print(f"Followed by that purchase: {hits / max(fires, 1):.0%}")
print(f"Chance of a purchase in a random hour and category: {base_rate:.0%}")
print(f"Lift over random: {(hits / max(fires, 1)) / base_rate:.1f}x")

# =========================================================
# STEP 8: Export the trained weights for Lovable
# Copy the printed JSON into your Lovable prompt.
# =========================================================
export = {cat: {"intercept": round(float(m.intercept_[0]), 4),
                "weights": {f: round(float(w), 4) for f, w in zip(FEATURES, m.coef_[0])}}
          for cat, m in habit_models.items()}
export["usual_hour"] = {f"{c}_{'weekday' if w else 'weekend'}": float(h) for (c, w), h in USUAL_HOUR.items()}
print("\nWEIGHTS FOR LOVABLE:")
print(json.dumps(export, indent=2))

# =========================================================
# STEP 9: Export Sarah's fake purchase history for the app
# The app uses this for weekly counts, "last bought" times, and history.
# Each purchase: [days_ago, hour, category, merchant]
# =========================================================
def app_merchant(category, hour, day):
    if category == "ride":    return "Lyft" if day % 2 else "Uber"
    if category == "coffee":  return "Starbucks"
    if category == "online":  return "Amazon"
    if category == "grocery": return "Grocery store"
    if hour >= 21:            return "Taco Bell" if day % 2 else "Burger King"
    if 11 <= hour <= 14:      return "Chick-fil-A" if day % 2 else "Chipotle"
    return "Chick-fil-A" if day % 3 else "McDonald's"

last_day = history["day"].max()
purchases = history[history["bought"] == 1].sort_values(["day", "hour"])
sarah_history = [[int(last_day - r.day), int(r.hour), r.category, app_merchant(r.category, r.hour, r.day)]
                for r in purchases.itertuples()
                if not (r.day == last_day and r.category == "ride")]   # today's ride home hasn't happened yet
print(f"\nSARAH'S HISTORY FOR LOVABLE ({len(sarah_history)} purchases over 10 weeks):")
print(json.dumps(sarah_history, separators=(",", ":")))

weekly = purchases[purchases["day"] > last_day - 28].groupby("category").size() / 4
print("\nAverage purchases per week (last 4 weeks):")
print(weekly.round(1).to_string())

Rows of history: 5600
category
coffee      77
food       119
grocery     24
online      14
ride        52
Name: purchases, dtype: int64

Sarah's usual visit times (hour of day):
  coffee   weekend: 13.0
  coffee   weekday: 9.0
  food     weekend: 15.0
  food     weekday: 13.0
  grocery  weekend: 15.0
  grocery  weekday: 17.0
  online   weekend: 15.0
  online   weekday: 20.0
  ride     weekend: 22.0
  ride     weekday: 15.0

How well each visit rate model ranks unseen hours (AUC: 0.5 = random guess, 1.0 = perfect)
  coffee   AUC = 0.74
  food     AUC = 0.82
  grocery  AUC = 0.73
  online   AUC = 1.00
  ride     AUC = 0.86

Tuesday 8 AM at the library -> NOTIFY: Free drink at Campus Brew
    merchant        reward  visit_rate  miles  location  points  match
 Campus Brew    Free drink        0.45    0.0       1.0    0.76  0.341
   Taco Bell  $8 off order        0.02    0.4       0.6    0.90  0.012
Campus Grill 15% off lunch        0.02    0.2       0.6    0.59  0.008
Online store   $10 vo